In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
df = pd.read_csv("processed_data/tickets_2M_final.csv")

required_cols = ["input_text", "category", "issue_type", "resolution"]
df = df[required_cols].copy()

df["input_text"] = df["input_text"].fillna("").astype(str)
df["category"] = df["category"].fillna("").astype(str)
df["issue_type"] = df["issue_type"].fillna("").astype(str)
df["resolution"] = df["resolution"].fillna("").astype(str)

print("Full dataset shape:", df.shape)

C:\Users\ricky\AppData\Local\Temp\ipykernel_16192\3151739789.py:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("processed_data/tickets_2M_final.csv")


Full dataset shape: (2000000, 4)


In [4]:
# Fast training ke liye sample lo
sample_size = 300000   # isko 400000 bhi kar sakte ho agar system handle kare
df_sample = df.sample(n=sample_size, random_state=42)

print("Sample dataset shape:", df_sample.shape)
df_sample.head()

Sample dataset shape: (300000, 4)


,input_text,category,issue_type,resolution
1828401,Cannot detect printer driver on tray Printer i...,Printer,Driver Missing Variant 52,Install or update the printer driver.
1200071,Documents not printing on queue Printer queue ...,Printer,Print Queue Stuck Variant 529,Clear print queue and restart print spooler se...
194849,VPN authentication failed on wan After resetti...,Network,VPN Authentication Failure Variant 5,"Clear saved VPN credentials, re-enter password..."
1629054,login for coming on not compliance otp otp is ...,Security,MFA Code Not Received Variant 105,Check registered device and resend MFA code.
191144,Laptop keyboard not working on monitor Several...,Hardware,Keyboard Not Working Variant 134,Reconnect keyboard driver or restart the device.


In [5]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [6]:
category_encoder = LabelEncoder()
issue_encoder = LabelEncoder()

df_sample["category_label"] = category_encoder.fit_transform(df_sample["category"])
df_sample["issue_label"] = issue_encoder.fit_transform(df_sample["issue_type"])

print("Unique categories:", len(category_encoder.classes_))
print("Unique issue types in sample:", len(issue_encoder.classes_))

Unique categories: 8
Unique issue types in sample: 5000


In [7]:
category_encoder = LabelEncoder()
issue_encoder = LabelEncoder()

df_sample["category_label"] = category_encoder.fit_transform(df_sample["category"])
df_sample["issue_label"] = issue_encoder.fit_transform(df_sample["issue_type"])

print("Unique categories:", len(category_encoder.classes_))
print("Unique issue types in sample:", len(issue_encoder.classes_))

Unique categories: 8
Unique issue types in sample: 5000


In [8]:
X = df_sample["input_text"]
y_category = df_sample["category_label"]
y_issue = df_sample["issue_label"]

X_train, X_test, y_cat_train, y_cat_test, y_issue_train, y_issue_test = train_test_split(
    X, y_category, y_issue,
    test_size=0.2,
    random_state=42,
    stratify=y_category
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 240000
Test size: 60000


In [9]:
tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("X_train_tfidf shape:", X_train_tfidf.shape)
print("X_test_tfidf shape:", X_test_tfidf.shape)

X_train_tfidf shape: (240000, 9436)
X_test_tfidf shape: (60000, 9436)


In [10]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score

In [11]:
# Category model
category_model = SGDClassifier(
    loss="log_loss",
    max_iter=20,
    random_state=42,
    n_jobs=-1
)

category_model.fit(X_train_tfidf, y_cat_train)

cat_pred = category_model.predict(X_test_tfidf)
cat_acc = accuracy_score(y_cat_test, cat_pred)

print("Category Accuracy:", round(cat_acc, 4))

Category Accuracy: 1.0


In [12]:
# issue lookup table
issue_lookup_df = df[["input_text", "issue_type", "resolution", "category"]].drop_duplicates().copy()

# har issue_type ka ek default resolution map
issue_resolution_map = (
    df.groupby("issue_type")["resolution"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])
    .to_dict()
)

print("Unique issue lookup rows:", issue_lookup_df.shape[0])
print("Unique issue->resolution mappings:", len(issue_resolution_map))

Unique issue lookup rows: 768122
Unique issue->resolution mappings: 5000


In [13]:
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
# category-wise issue lookup
category_issue_lookup = {}

for cat in issue_lookup_df["category"].unique():
    temp = issue_lookup_df[issue_lookup_df["category"] == cat].copy()
    temp = temp.drop_duplicates(subset=["issue_type"])
    category_issue_lookup[cat] = temp.reset_index(drop=True)

print("Category-wise lookup prepared.")

Category-wise lookup prepared.


In [15]:
def predict_ticket(ticket_text):
    # 1) category predict
    text_vec = tfidf.transform([ticket_text])
    pred_cat_label = category_model.predict(text_vec)[0]
    pred_category = category_encoder.inverse_transform([pred_cat_label])[0]

    # 2) issue lookup only inside predicted category
    candidate_df = category_issue_lookup[pred_category].copy()

    candidate_vectors = tfidf.transform(candidate_df["input_text"])
    sims = cosine_similarity(text_vec, candidate_vectors).flatten()

    best_idx = sims.argmax()
    pred_issue = candidate_df.iloc[best_idx]["issue_type"]

    # 3) resolution
    pred_resolution = issue_resolution_map.get(pred_issue, "No solution found")

    return {
        "predicted_category": pred_category,
        "predicted_issue_type": pred_issue,
        "suggested_solution": pred_resolution
    }

In [16]:
# sample test
sample_ticket_1 = "Outlook keeps asking for password again and again. Mail is not syncing."
sample_ticket_2 = "VPN connection failed and authentication error is showing."
sample_ticket_3 = "Printer is offline and documents are stuck in queue."

print(predict_ticket(sample_ticket_1))
print(predict_ticket(sample_ticket_2))
print(predict_ticket(sample_ticket_3))

{'predicted_category': 'Email', 'predicted_issue_type': 'Outlook Password Prompt Loop Variant 282', 'suggested_solution': 'Clear saved Outlook credentials and sign in again.'}
{'predicted_category': 'Network', 'predicted_issue_type': 'VPN Authentication Failure Variant 445', 'suggested_solution': 'Clear saved VPN credentials, re-enter password, and reconnect.'}
{'predicted_category': 'Printer', 'predicted_issue_type': 'Printer Offline Variant 158', 'suggested_solution': 'Restart printer and verify network connection.'}


In [17]:
import joblib

# Models ko Python 3.14 compatible save karo
joblib.dump(category_model, "saved_models/category_model.pkl", compress=3)
joblib.dump(tfidf, "saved_models/tfidf_vectorizer.pkl", compress=3)
joblib.dump(category_encoder, "saved_models/category_encoder.pkl", compress=3)
joblib.dump(issue_resolution_map, "saved_models/issue_resolution_map.pkl", compress=3)
joblib.dump(category_issue_lookup, "saved_models/category_issue_lookup.pkl", compress=3)

print("✅ Models saved successfully!")

✅ Models saved successfully!


In [19]:
import pickle
import joblib

# Pehle joblib se load karo (agar already save hai)
category_model = joblib.load("saved_models/category_model.pkl")
tfidf = joblib.load("saved_models/tfidf_vectorizer.pkl")
category_encoder = joblib.load("saved_models/category_encoder.pkl")
issue_resolution_map = joblib.load("saved_models/issue_resolution_map.pkl")
category_issue_lookup = joblib.load("saved_models/category_issue_lookup.pkl")

# Ab pickle se save karo (overwrite)
with open("saved_models/category_model.pkl", "wb") as f:
    pickle.dump(category_model, f, protocol=4)

with open("saved_models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f, protocol=4)

with open("saved_models/category_encoder.pkl", "wb") as f:
    pickle.dump(category_encoder, f, protocol=4)

with open("saved_models/issue_resolution_map.pkl", "wb") as f:
    pickle.dump(issue_resolution_map, f, protocol=4)

with open("saved_models/category_issue_lookup.pkl", "wb") as f:
    pickle.dump(category_issue_lookup, f, protocol=4)

print("✅ All models saved with pickle (protocol=4)!")

✅ All models saved with pickle (protocol=4)!
